# 06. Движок скоринга: PD → рейтинг → лимит/ставка → флаги политики → причины

ТЗ §2.5 (логика ответа `/score`; HTTP-слой FastAPI — отдельная тонкая обёртка над этим движком).
**Граница ответственности:** движок отдаёт *оценку риска и рекомендацию* (`pd`, `riskScore`, `ratingGrade`, `recommendedLimit/Rate`, `policyFlags`, `reasons`).
Решение об одобрении принимает `credit-core` — пороги одобрения и бизнес-правила банка сюда **не зашиты**.

Состав ответа: `pd`, `riskScore`, `ratingGrade`, `riskSegment`, `recommendedLimit`, `recommendedRate`, `maxTermMonths`, `policyFlags[]`, `reasons[]`, `warnings[]`, `modelVersion`, `calculatedAt`, `latencyMs`.

## 1. Модуль `engine.py`
**Формулы рекомендации** (описываются в документации):
- доля EBITDA, доступная под новый платёж: `share = clip(0.65 − 1.5·PD, 0, 0.6)`;
- платёжная способность: `capacity = max(0, share · EBITDA_мес − текущие платежи по БКИ)`;
- лимит по денежному потоку: приведённая стоимость аннуитета `capacity` при ставке и сроке рекомендации;
- лимит по оборотам: `k_продукта · средний месячный оборот` (+до 25% при обеспечении);
- `recommendedLimit = min(лимит_CF, лимит_оборотов, потолок_продукта)`, округление вниз до 10 тыс. ₽, меньше минимального лимита → 0;
- ставка: `база_продукта + min(0.5% + 0.45·PD, 12%)`; максимальный срок: `макс_продукта · clip(1 − 1.2·PD, 0.4, 1)` с округлением до 6 мес.

In [1]:
%%writefile ../risk-scoring/riskscore/engine.py
"""Движок скоринга: оценка риска + рекомендация + флаги политики + объяснение. Детерминирован.

Данные и модель синтетические. Движок НЕ принимает кредитное решение (это зона credit-core).
"""
from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from .features import FEATURE_META, compute_features

# --- продуктовые параметры рекомендации (конфигурация, а не пороги одобрения)
PRODUCT_CONFIG = {
    #                  база ставки, макс. срок, k * средний месячный оборот, потолок лимита, мин. лимит
    "OVERDRAFT":       dict(rate=0.205, max_term=12, k_turnover=0.35, cap=15_000_000, min_limit=100_000),
    "WORKING_CAPITAL": dict(rate=0.190, max_term=36, k_turnover=1.50, cap=40_000_000, min_limit=100_000),
    "INVESTMENT":      dict(rate=0.175, max_term=60, k_turnover=3.00, cap=50_000_000, min_limit=300_000),
    "EXPRESS":         dict(rate=0.240, max_term=24, k_turnover=0.50, cap=3_000_000, min_limit=100_000),
}
DEFAULT_PRODUCT = "WORKING_CAPITAL"

INDUSTRY_RU = {
    "retail_food": "розничная торговля продуктами", "catering": "общественное питание", "construction": "строительство",
    "transport": "транспорт и логистика", "it_services": "IT-услуги", "manufacturing": "производство",
    "wholesale": "оптовая торговля", "agriculture": "сельское хозяйство", "personal_serv": "бытовые услуги",
    "health_edu": "медицина и образование", "real_estate": "аренда недвижимости",
}

# код: (важность, описание). Флаги НЕ являются решением: их интерпретирует credit-core.
FLAG_DEFS = {
    "LIQUIDATION_IN_PROGRESS": ("CRITICAL", "Компания находится в процессе ликвидации"),
    "ACCOUNT_BLOCKED": ("CRITICAL", "Есть блокировка расчётных счетов"),
    "DELINQUENCY_90_PLUS": ("CRITICAL", "В кредитной истории есть просрочка 90+ дней"),
    "DISQUALIFIED_DIRECTOR": ("CRITICAL", "Среди руководителей есть дисквалифицированное лицо"),
    "BANKRUPTCY_SIGNS": ("CRITICAL", "Обнаружены сведения о процедуре банкротства"),
    "TAX_DEBT_SIGNIFICANT": ("WARNING", "Значимая задолженность по налогам"),
    "CURRENT_OVERDUE": ("WARNING", "Есть текущая просроченная задолженность по кредитам"),
    "DEFAULT_IN_HISTORY": ("WARNING", "В кредитной истории есть дефолты"),
    "NO_CREDIT_HISTORY": ("WARNING", "Кредитная история отсутствует: оценка менее надёжна"),
    "MASS_ADDRESS": ("WARNING", "Адрес регистрации помечен как массовый"),
    "MASS_DIRECTOR": ("WARNING", "Руководитель числится в большом числе компаний"),
    "YOUNG_BUSINESS": ("WARNING", "Бизнесу менее 6 месяцев"),
    "NEGATIVE_EQUITY": ("WARNING", "Отрицательный собственный капитал"),
    "HIGH_DEBT_BURDEN": ("WARNING", "Совокупные платежи превышают месячную EBITDA (DSCR < 1)"),
    "FREQUENT_INQUIRIES": ("INFO", "Много кредитных запросов за последние 3 месяца"),
    "TURNOVER_MISMATCH": ("INFO", "Обороты по счёту заметно расходятся с выручкой из отчётности"),
}
# признаки, значения которых показываем в процентах
_PCT = {"net_margin", "gross_margin", "ebitda_margin", "roa", "equity_ratio", "revenue_growth_yoy", "revenue_growth_last2q", "revenue_cv",
        "turnover_cv", "turnover_trend", "revenue_trend", "new_payment_to_ebitda", "existing_payment_to_ebitda", "payment_burden",
        "liabilities_to_revenue", "debt_to_revenue", "balance_to_turnover", "tax_debt_ratio", "turnover_to_revenue", "amount_to_revenue",
        "own_contribution_pct", "collateral_cover"}
_DAYS = {"payables_days", "receivables_days", "max_overdue_days"}
_MONTHS = {"company_age_months", "history_months", "bank_tenure_months"}


def _fmt(code: str, v) -> str:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "нет данных"
    if isinstance(v, str):
        return INDUSTRY_RU.get(v, v)
    if code in _PCT:
        return f"{v * 100:.1f}%"
    if code in _DAYS:
        return f"{v:.0f} дн."
    if code in _MONTHS:
        return f"{v:.0f} мес."
    if code.endswith("_log"):
        return f"{np.exp(v):,.0f}".replace(",", " ")
    if float(v).is_integer():
        return f"{v:.0f}"
    return f"{v:.2f}"


class ScoringEngine:
    def __init__(self, models_dir: str | Path, model_file: str = "risk_model_woe_logit_v1.joblib"):
        d = Path(models_dir)
        self.model = joblib.load(d / model_file)
        pipe = self.model.pipeline
        self.clip, self.enc, self.sel = pipe.named_steps["clip"], pipe.named_steps["woe"], pipe.named_steps["select"]
        self.tail = pipe[2:]                              # WOE -> отбор -> логрегрессия
        self.card = json.loads((d / "scorecard_v1.json").read_text(encoding="utf-8"))
        self.scale = json.loads((d / "rating_scale_v1.json").read_text(encoding="utf-8"))["grades"]
        self.grades = [g["grade"] for g in self.scale]
        self.upper = [g["pd_to"] for g in self.scale]
        self.segment = {g["grade"]: g["segment"] for g in self.scale}
        self.version = self.model.version
        self.cols = list(self.card["features"])
        self.pts = {c: np.array(v["points"]) for c, v in self.card["features"].items()}
        self.ref = np.array([self.card["features"][c]["ref_points"] for c in self.cols])

    # ------------------------------------------------------------------ вспомогательное
    def _pd_and_points(self, Xc: pd.DataFrame):
        p = self.model.calibrate(self.tail.predict_proba(Xc)[:, 1])
        pts = np.column_stack([self.pts[c][np.minimum(self.enc.bin_index(Xc, c), len(self.pts[c]) - 1)] for c in self.cols])
        return p, pts

    def grade_of(self, p) -> np.ndarray:
        return np.array(self.grades)[np.searchsorted(self.upper, p, side="right").clip(0, len(self.grades) - 1)]

    def _flags(self, raw: pd.DataFrame, X: pd.DataFrame) -> list[list[dict]]:
        rev_ltm = np.exp(X["revenue_ltm_log"].to_numpy())
        tax_debt = pd.to_numeric(raw.get("fns_tax_debt", pd.Series(0.0, index=raw.index)), errors="coerce").fillna(0).to_numpy()
        cond = {
            "LIQUIDATION_IN_PROGRESS": X.is_liquidating.eq(1),
            "ACCOUNT_BLOCKED": X.account_blocked.eq(1),
            "DELINQUENCY_90_PLUS": X.overdue_90_cnt.gt(0) | X.max_overdue_days.ge(90),
            "DISQUALIFIED_DIRECTOR": X.disqualified.eq(1),
            "BANKRUPTCY_SIGNS": X.bankruptcy.eq(1),
            "TAX_DEBT_SIGNIFICANT": pd.Series(tax_debt >= np.maximum(100_000, 0.02 * rev_ltm), index=X.index),
            "CURRENT_OVERDUE": X.current_overdue_flag.eq(1),
            "DEFAULT_IN_HISTORY": X.defaults_cnt.gt(0),
            "NO_CREDIT_HISTORY": X.has_credit_history.eq(0),
            "MASS_ADDRESS": X.mass_address.eq(1),
            "MASS_DIRECTOR": X.mass_director.eq(1),
            "YOUNG_BUSINESS": X.company_age_months.lt(6),
            "NEGATIVE_EQUITY": X.negative_equity.eq(1),
            "HIGH_DEBT_BURDEN": X.payment_burden.gt(1.0),
            "FREQUENT_INQUIRIES": X.inquiries_3m.ge(4),
            "TURNOVER_MISMATCH": X.turnover_to_revenue.lt(0.5) | X.turnover_to_revenue.gt(1.6),
        }
        M = np.column_stack([cond[k].fillna(False).to_numpy(bool) for k in FLAG_DEFS])
        keys = list(FLAG_DEFS)
        return [[dict(code=keys[j], severity=FLAG_DEFS[keys[j]][0], description=FLAG_DEFS[keys[j]][1]) for j in np.where(r)[0]] for r in M]

    def _offer(self, raw: pd.DataFrame, X: pd.DataFrame, p: np.ndarray):
        n = len(raw)
        prod = raw.get("product_code", pd.Series(DEFAULT_PRODUCT, index=raw.index)).astype(object)
        known = prod.isin(PRODUCT_CONFIG).to_numpy()
        prod_eff = prod.where(known, DEFAULT_PRODUCT)
        cfg = lambda k: prod_eff.map(lambda x: PRODUCT_CONFIG[x][k]).to_numpy(float)
        ebitda_m = pd.to_numeric(raw[[f"ebitda_q{k}" for k in range(5, 9)]].stack(future_stack=True).groupby(level=0).sum(min_count=4), errors="coerce").reindex(raw.index).to_numpy() / 12 \
            if all(f"ebitda_q{k}" in raw for k in range(5, 9)) else np.full(n, np.nan)
        existing = np.nan_to_num(pd.to_numeric(raw.get("bki_monthly_payment", pd.Series(np.nan, index=raw.index)), errors="coerce").to_numpy(float), nan=0.0)
        turn = np.exp(X["turnover_avg_log"].to_numpy())
        cover = np.nan_to_num(X["collateral_cover"].to_numpy(), nan=0.0)
        term_req = pd.to_numeric(raw.get("term_months", pd.Series(np.nan, index=raw.index)), errors="coerce").to_numpy(float)

        share = np.clip(0.65 - 1.5 * p, 0.0, 0.6)
        rate = cfg("rate") + np.minimum(0.005 + 0.45 * p, 0.12)
        max_term = np.maximum(np.round(cfg("max_term") * np.clip(1 - 1.2 * p, 0.4, 1.0) / 6) * 6, 6)
        max_term = np.minimum(max_term, cfg("max_term"))
        term_ref = np.where(np.isnan(term_req), max_term, np.minimum(np.maximum(term_req, 1), max_term))
        capacity = np.maximum(0.0, share * np.nan_to_num(ebitda_m, nan=0.0) - existing)
        r = rate / 12
        limit_cf = capacity * (1 - (1 + r) ** (-term_ref)) / r
        limit_t = cfg("k_turnover") * turn * (1 + 0.25 * np.minimum(cover, 1.0))
        limit = np.minimum.reduce([limit_cf, limit_t, cfg("cap")])
        limit = np.floor(limit / 10_000) * 10_000
        limit = np.where((limit < cfg("min_limit")) | ~(np.nan_to_num(ebitda_m, nan=-1) > 0), 0.0, limit)
        return limit, rate, max_term, known, capacity

    def _reasons(self, X: pd.DataFrame, Xc: pd.DataFrame, pts: np.ndarray, n_up=4, n_down=2) -> list[list[dict]]:
        contrib = pts - self.ref                          # баллы относительно среднего заёмщика (минус = хуже)
        out = []
        for i in range(len(X)):
            c = contrib[i]
            up = [j for j in np.argsort(c) if c[j] < -0.5][:n_up]            # повышают риск
            down = [j for j in np.argsort(-c) if c[j] > 0.5][:n_down]        # снижают риск
            rest = [j for j in np.argsort(-np.abs(c)) if j not in up + down]
            chosen = up + down
            chosen += rest[: max(0, 5 - len(chosen))]
            chosen.sort(key=lambda j: -abs(c[j]))
            rs = []
            for j in chosen:
                code = self.cols[j]; v = X[code].iloc[i]
                risk_up = c[j] < 0
                rs.append(dict(
                    featureCode=code, featureLabel=FEATURE_META[code]["label"],
                    direction="INCREASES_RISK" if risk_up else "DECREASES_RISK", contribution=round(float(c[j]), 1),
                    explanation=f"{FEATURE_META[code]['label']}: {_fmt(code, v)}. "
                                f"{'Повышает' if risk_up else 'Снижает'} оценку риска ({c[j]:+.0f} баллов к скору относительно среднего заёмщика)."))
            out.append(rs)
        return out

    # ------------------------------------------------------------------ публичный API
    def score_frame(self, raw: pd.DataFrame, calculated_at: str | None = None) -> list[dict]:
        t0 = time.perf_counter()
        raw = raw.reset_index(drop=True)
        X = compute_features(raw)
        Xc = self.clip.transform(X)
        p, pts = self._pd_and_points(Xc)
        grade = self.grade_of(p)
        score = np.clip(np.round(self.card["base_points"] + pts.sum(1)), self.card["score_min"], self.card["score_max"])
        limit, rate, max_term, known, capacity = self._offer(raw, X, p)
        flags, reasons = self._flags(raw, X), self._reasons(X, Xc, pts)
        clipped = self.clip.clipped_mask(X)
        key_inputs = ["revenue_q8", "ebitda_q8", "total_assets_q8", "requested_amount", "term_months", "application_date", "reg_date"]
        now = calculated_at or datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        res = []
        for i in range(len(raw)):
            w = [f"FEATURE_CLIPPED:{c}" for c in clipped.columns[clipped.iloc[i].to_numpy(bool)]]
            w += [f"INPUT_MISSING:{k}" for k in key_inputs if k not in raw or pd.isna(raw[k].iloc[i])]
            if not known[i]:
                w.append(f"UNKNOWN_PRODUCT:{raw['product_code'].iloc[i] if 'product_code' in raw else None}->{DEFAULT_PRODUCT}")
            res.append(dict(
                pd=round(float(p[i]), 5), riskScore=int(score[i]), ratingGrade=str(grade[i]), riskSegment=self.segment[str(grade[i])],
                recommendedLimit=int(limit[i]), recommendedRate=round(float(rate[i]), 4), maxTermMonths=int(max_term[i]),
                policyFlags=flags[i], reasons=reasons[i], warnings=w, modelVersion=self.version, calculatedAt=now, latencyMs=0.0))
        per_row = (time.perf_counter() - t0) * 1000 / max(len(raw), 1)
        for r in res:
            r["latencyMs"] = round(per_row, 2)
        return res

    def score_one(self, raw_row: dict, calculated_at: str | None = None) -> dict:
        return self.score_frame(pd.DataFrame([raw_row]), calculated_at)[0]

    def suggest_conditions(self, raw_row: dict, factors=(1.0, .9, .8, .7, .6, .5, .4, .3, .2)) -> list[dict]:
        """What-if: при каких сумме/сроке рекомендованный лимит покрывает запрос и нет критических флагов."""
        prod = raw_row.get("product_code", DEFAULT_PRODUCT)
        prod = prod if prod in PRODUCT_CONFIG else DEFAULT_PRODUCT
        max_t = PRODUCT_CONFIG[prod]["max_term"]
        terms = sorted({t for t in (6, 12, 18, 24, 36, 48, 60) if t <= max_t} | {int(raw_row.get("term_months", 12))})
        rows = [dict(raw_row, requested_amount=round(float(raw_row["requested_amount"]) * f, -4), term_months=t) for t in terms for f in factors]
        res = self.score_frame(pd.DataFrame(rows), calculated_at="1970-01-01T00:00:00Z")
        out = []
        for t in terms:
            ok = [(r, rw) for r, rw in zip(res, rows) if rw["term_months"] == t and r["recommendedLimit"] >= rw["requested_amount"] > 0
                  and not any(f["severity"] == "CRITICAL" for f in r["policyFlags"])]
            if ok:
                r, rw = max(ok, key=lambda z: z[1]["requested_amount"])
                out.append(dict(termMonths=t, maxAmount=rw["requested_amount"], pd=r["pd"], ratingGrade=r["ratingGrade"], recommendedRate=r["recommendedRate"]))
        return out

    def model_info(self, metrics_path: str | Path | None = None) -> dict:
        info = dict(modelVersion=self.version, trainedAt=self.model.meta.get("trained_at"), datasetVersion=self.model.meta.get("dataset_version"),
                    features=[dict(code=c, label=FEATURE_META[c]["label"], group=FEATURE_META[c]["group"]) for c in self.cols],
                    allFeaturesCount=len(FEATURE_META), ratingScale=self.scale, scoreScale=dict(base=self.card["base_score"], pdAtBase=self.card["pd_ref"], pdo=self.card["pdo"], min=self.card["score_min"], max=self.card["score_max"]),
                    note="Модель обучена на синтетических данных; метрики не переносятся на реальный портфель.")
        if metrics_path and Path(metrics_path).exists():
            m = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
            t = m["metrics"]["WOE-LR + скор-карта (основная)|test"]
            info["metrics"] = dict(gini=t["gini"], ks=t["ks"], brier=t["brier"], auc=t["auc"], psiScoreTrainTest=m["psi"]["score_train_test"], sample="test")
        return info

Overwriting ../risk-scoring/riskscore/engine.py


## 2. Подключение движка и первый скоринг

In [2]:
import sys, json, time, warnings, subprocess
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.engine import ScoringEngine, PRODUCT_CONFIG, FLAG_DEFS
from riskscore import metrics as M
STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}
raw = pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype=STR)
split = pd.read_csv(ROOT / "data/generated/split.csv").split.to_numpy()
y = raw.default_flag.astype(int).to_numpy()
eng = ScoringEngine(ROOT / "risk-scoring/models")
row = raw.iloc[5].to_dict()
r = eng.score_one(row, calculated_at="2026-10-26T12:00:00Z")
print(json.dumps({k: v for k, v in r.items() if k not in ("reasons", "policyFlags")}, ensure_ascii=False, indent=1))
print("policyFlags:", [f["code"] for f in r["policyFlags"]])
for x in r["reasons"]: print(f"  [{x['direction']:14s}] {x['contribution']:+6.1f}  {x['explanation']}")

{
 "pd": 0.12643,
 "riskScore": 527,
 "ratingGrade": "CCC",
 "riskSegment": "HIGH",
 "recommendedLimit": 210000,
 "recommendedRate": 0.2519,
 "maxTermMonths": 30,
 "warnings": [],
 "modelVersion": "v1.0.0-woe-logit",
 "calculatedAt": "2026-10-26T12:00:00Z",
 "latencyMs": 73.71
}
policyFlags: ['NO_CREDIT_HISTORY']
  [INCREASES_RISK]  -24.0  Рост выручки г/г: -26.9%. Повышает оценку риска (-24 баллов к скору относительно среднего заёмщика).
  [INCREASES_RISK]  -13.4  Запросов в БКИ за 12 мес.: 9. Повышает оценку риска (-13 баллов к скору относительно среднего заёмщика).
  [INCREASES_RISK]  -10.0  Убыточных кварталов из 8: 2. Повышает оценку риска (-10 баллов к скору относительно среднего заёмщика).
  [INCREASES_RISK]   -8.6  Волатильность выручки: 22.5%. Повышает оценку риска (-9 баллов к скору относительно среднего заёмщика).
  [DECREASES_RISK]   +5.8  Коэффициент автономии: 33.2%. Снижает оценку риска (+6 баллов к скору относительно среднего заёмщика).
  [DECREASES_RISK]   +2.7  Валова

## 3. Один профиль × четыре продукта → разные рекомендации (ТЗ 2.5.6)

In [3]:
te_ix = np.where(split == "test")[0]
rs_ = eng.score_frame(raw.iloc[te_ix].assign(product_code="WORKING_CAPITAL", term_months=24, requested_amount=1_000_000))
best_i = int(np.argmax([r["recommendedLimit"] if not any(f["severity"] == "CRITICAL" for f in r["policyFlags"]) and r["ratingGrade"] in ("A", "BBB") else -1 for r in rs_]))
base = dict(raw.iloc[te_ix[best_i]].to_dict(), requested_amount=1_500_000)
rows = []
for prod in PRODUCT_CONFIG:
    z = dict(base, product_code=prod, indicative_rate=PRODUCT_CONFIG[prod]["rate"], term_months={"OVERDRAFT": 12, "WORKING_CAPITAL": 24, "INVESTMENT": 48, "EXPRESS": 12}[prod])
    rr = eng.score_one(z, "2026-10-26T12:00:00Z"); rows.append(dict(product=prod, requested=z["requested_amount"], term=z["term_months"], pd=rr["pd"], grade=rr["ratingGrade"],
                                     limit=rr["recommendedLimit"], rate=rr["recommendedRate"], maxTerm=rr["maxTermMonths"]))
print(pd.DataFrame(rows).to_string(index=False))

        product  requested  term      pd grade    limit   rate  maxTerm
      OVERDRAFT    1500000    12 0.04407     A  7500000 0.2298       12
WORKING_CAPITAL    1500000    24 0.04407     A 28150000 0.2148       36
     INVESTMENT    1500000    48 0.04407     A 47770000 0.1998       54
        EXPRESS    1500000    12 0.04407     A  3000000 0.2648       24


## 4. Что-если: какие условия подходят (ТЗ 3.1, SHOULD)
Для заявки с лимитом меньше запроса — находим максимальную сумму при каждом сроке, под которую рекомендованный лимит покрывает запрос.

In [4]:
all_res = eng.score_frame(raw.iloc[np.where(split == "test")[0]])
lim_ratio = np.array([r["recommendedLimit"] / max(rw, 1) for r, rw in zip(all_res, raw.iloc[np.where(split == "test")[0]].requested_amount)])
cand = np.where((lim_ratio > 0.2) & (lim_ratio < 0.7))[0][0]
rw = raw.iloc[np.where(split == "test")[0][cand]].to_dict()
print(f"заявка: {rw['product_code']}, запрошено {rw['requested_amount']:,.0f} ₽ на {rw['term_months']} мес.; рекомендованный лимит {all_res[cand]['recommendedLimit']:,.0f} ₽")
print(pd.DataFrame(eng.suggest_conditions(rw)).to_string(index=False))

заявка: INVESTMENT, запрошено 820,000 ₽ на 24 мес.; рекомендованный лимит 560,000 ₽
 termMonths  maxAmount      pd ratingGrade  recommendedRate
         12   250000.0 0.01796         AAA           0.1881
         18   410000.0 0.01796         AAA           0.1881
         24   490000.0 0.01796         AAA           0.1881
         36   740000.0 0.01796         AAA           0.1881
         48   820000.0 0.01796         AAA           0.1881
         60   820000.0 0.01796         AAA           0.1881


## 5. Производительность (ТЗ 2.5.11: p95 < 300 мс на одиночный скоринг) и батч

In [5]:
sample = raw.sample(300, random_state=1).to_dict("records")
lat = []
for rw in sample:
    t = time.perf_counter(); eng.score_one(rw); lat.append((time.perf_counter() - t) * 1000)
print(f"одиночный скоринг, 300 вызовов: p50 {np.percentile(lat, 50):.1f} мс | p95 {np.percentile(lat, 95):.1f} мс | max {max(lat):.1f} мс  (требование p95 < 300 мс)")
t = time.perf_counter(); br = eng.score_frame(raw.iloc[:2000]); dt_ = time.perf_counter() - t
print(f"батч 2000 заявок: {dt_:.1f} с → {dt_ / 2000 * 1000:.2f} мс на заявку")
print("загрузка модели при старте:", end=" "); t = time.perf_counter(); ScoringEngine(ROOT / "risk-scoring/models"); print(f"{(time.perf_counter() - t) * 1000:.0f} мс (один раз)")

одиночный скоринг, 300 вызовов: p50 72.0 мс | p95 74.9 мс | max 142.2 мс  (требование p95 < 300 мс)


батч 2000 заявок: 0.4 с → 0.21 мс на заявку
загрузка модели при старте: 21 мс (один раз)


## 6. Статистика по рекомендациям на тесте (sanity check формул лимита)

In [6]:
te = np.where(split == "test")[0]
d = pd.DataFrame({"grade": [r["ratingGrade"] for r in all_res], "limit": [r["recommendedLimit"] for r in all_res],
                  "rate": [r["recommendedRate"] for r in all_res], "req": raw.iloc[te].requested_amount.to_numpy(),
                  "flags_crit": [any(f["severity"] == "CRITICAL" for f in r["policyFlags"]) for r in all_res]})
g = d.groupby("grade").agg(n=("limit", "size"), median_limit=("limit", "median"), share_zero_limit=("limit", lambda s: (s == 0).mean()), mean_rate=("rate", "mean"),
                           covers_request=("limit", lambda s: np.nan)).reindex(["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C", "D"])
g["covers_request"] = d.assign(c=d.limit >= d.req).groupby("grade").c.mean()
print(g.round(3).to_string())
print("\nдоля заявок с хотя бы одним CRITICAL-флагом:", round(d.flags_crit.mean(), 3))

         n  median_limit  share_zero_limit  mean_rate  covers_request
grade                                                                
AAA    352     1570000.0             0.031      0.211           0.858
AA     493      990000.0             0.089      0.215           0.698
A      375      620000.0             0.184      0.220           0.557
BBB    262      340000.0             0.328      0.228           0.378
BB     198           0.0             0.561      0.237           0.202
B      151           0.0             0.728      0.252           0.119
CCC    135           0.0             0.830      0.273           0.067
CC     195           0.0             0.959      0.303           0.000
C      128           0.0             0.992      0.318           0.000
D      111           0.0             1.000      0.318           0.000

доля заявок с хотя бы одним CRITICAL-флагом: 0.06


## 7. Автотесты (pytest): пайплайн, контракт ответа, граничные случаи, монотонность, детерминированность
Требование ТЗ — ≥ 15 тестов. Файл `risk-scoring/tests/test_scoring.py` запускается командой `pytest risk-scoring/tests`.

In [7]:
%%writefile ../risk-scoring/tests/test_scoring.py
"""Тесты пайплайна признаков, движка скоринга и генератора синтетики (все данные синтетические)."""
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

ROOT = Path(__file__).resolve().parents[2]
sys.path[:0] = [str(ROOT / "risk-scoring"), str(ROOT)]
from data.generate import is_valid_inn, is_valid_ogrn, is_synthetic_id          # noqa: E402
from riskscore.engine import ScoringEngine, PRODUCT_CONFIG, FLAG_DEFS             # noqa: E402
from riskscore.features import FEATURE_META, FEATURE_COLUMNS, compute_features    # noqa: E402

STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}


@pytest.fixture(scope="module")
def raw():
    return pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype=STR).iloc[:600]


@pytest.fixture(scope="module")
def eng():
    return ScoringEngine(ROOT / "risk-scoring/models")


def _strip(r):
    return {k: v for k, v in r.items() if k not in ("latencyMs", "calculatedAt")}


# ------------------------------------------------------------------ данные и признаки
def test_inn_ogrn_checksums_and_synthetic_range(raw):
    assert raw.inn.map(is_valid_inn).all() and raw.ogrn.map(is_valid_ogrn).all()
    assert raw.inn.map(is_synthetic_id).all()
    assert raw.inn.is_unique


def test_balance_identity(raw):
    for k in range(1, 9):
        assert (raw[f"total_assets_q{k}"] - raw[f"total_liabilities_q{k}"] - raw[f"equity_q{k}"]).abs().max() <= 2


def test_at_least_25_features_in_5_groups():
    assert len(FEATURE_COLUMNS) >= 25
    assert len({m["group"] for m in FEATURE_META.values()}) == 5


def test_features_deterministic(raw):
    assert compute_features(raw).equals(compute_features(raw))


def test_missing_columns_and_garbage_do_not_crash(raw):
    bad = raw.iloc[:3].drop(columns=["turnover_credit_m5", "balance_end_m3", "revenue_q1"]).copy()
    bad.loc[bad.index[0], "application_date"] = "не дата"
    bad.loc[bad.index[1], "requested_amount"] = "abc"
    X = compute_features(bad)
    assert X.shape == (3, len(FEATURE_COLUMNS)) and not np.isinf(X.select_dtypes("number").to_numpy()).any()


def test_no_credit_history_is_nan_not_zero(raw):
    nh = raw[raw.bki_has_history == 0].iloc[:50]
    X = compute_features(nh)
    assert (X.has_credit_history == 0).all() and X.overdue_90_cnt.isna().all() and X.open_loans.isna().all()


# ------------------------------------------------------------------ контракт ответа
def test_response_contract_fields(eng, raw):
    r = eng.score_one(raw.iloc[0].to_dict())
    for k in ("pd", "riskScore", "ratingGrade", "riskSegment", "recommendedLimit", "recommendedRate", "maxTermMonths",
              "policyFlags", "reasons", "modelVersion", "calculatedAt", "latencyMs", "warnings"):
        assert k in r
    assert 0 < r["pd"] < 1 and 300 <= r["riskScore"] <= 900


def test_reasons_at_least_5_with_required_fields(eng, raw):
    for i in range(0, 600, 60):
        rs = eng.score_one(raw.iloc[i].to_dict())["reasons"]
        assert len(rs) >= 5
        for x in rs:
            assert {"featureCode", "featureLabel", "direction", "contribution", "explanation"} <= set(x)
            assert x["direction"] in ("INCREASES_RISK", "DECREASES_RISK") and len(x["explanation"]) > 10


def test_deterministic_inference(eng, raw):
    a, b = eng.score_one(raw.iloc[7].to_dict()), eng.score_one(raw.iloc[7].to_dict())
    assert _strip(a) == _strip(b)


def test_batch_equals_single(eng, raw):
    batch = eng.score_frame(raw.iloc[:25])
    for i in (0, 11, 24):
        assert _strip(batch[i]) == _strip(eng.score_one(raw.iloc[i].to_dict()))


def test_sum_of_points_matches_riskscore(eng, raw):
    X = eng.clip.transform(compute_features(raw.iloc[:200]))
    p, pts = eng._pd_and_points(X)
    score = np.clip(np.round(eng.card["base_points"] + pts.sum(1)), 300, 900)
    direct = np.clip(np.round(600 - eng.card["factor"] * (np.log(p / (1 - p)) - np.log(0.05 / 0.95))), 300, 900)
    inside = (p > 0.0021) & (p < 0.949)
    assert np.abs(score - direct)[inside].max() <= 1


def test_rating_scale_is_monotone_in_pd(eng):
    ps = np.linspace(0.002, 0.95, 400)
    order = {g: i for i, g in enumerate(eng.grades)}
    idx = [order[g] for g in eng.grade_of(ps)]
    assert idx == sorted(idx) and len(set(idx)) == len(eng.grades) and len(eng.grades) >= 7


# ------------------------------------------------------------------ монотонность и бизнес-логика
def test_pd_not_decreasing_with_larger_amount(eng, raw):
    base = raw.iloc[:200]
    p0 = np.array([r["pd"] for r in eng.score_frame(base)])
    z = base.copy(); z["requested_amount"] = z.requested_amount * 2
    p1 = np.array([r["pd"] for r in eng.score_frame(z)])
    assert (p1 >= p0 - 1e-9).all()


def test_pd_not_decreasing_with_more_existing_debt_payment(eng, raw):
    base = raw[raw.bki_has_history == 1].iloc[:200]
    p0 = np.array([r["pd"] for r in eng.score_frame(base)])
    z = base.copy(); z["bki_monthly_payment"] = z.bki_monthly_payment * 2 + 20_000
    p1 = np.array([r["pd"] for r in eng.score_frame(z)])
    assert (p1 >= p0 - 1e-9).all()


def test_pd_not_decreasing_with_delinquency(eng, raw):
    base = raw[raw.bki_has_history == 1].iloc[:300]
    p0 = np.array([r["pd"] for r in eng.score_frame(base)])
    z = base.copy(); z["bki_overdue_90_cnt"] = 2; z["bki_overdue_60_cnt"] = 2; z["bki_overdue_30_cnt"] = 3; z["bki_max_overdue_days"] = 150
    p1 = np.array([r["pd"] for r in eng.score_frame(z)])
    assert (p1 >= p0 - 1e-9).all() and p1.mean() > p0.mean() + 0.01


def test_pd_not_decreasing_with_blocking(eng, raw):
    base = raw.iloc[:200]
    p0 = np.array([r["pd"] for r in eng.score_frame(base)])
    z = base.copy(); z["fns_account_blocked"] = 1
    p1 = np.array([r["pd"] for r in eng.score_frame(z)])
    assert (p1 >= p0 - 1e-9).all() and p1.mean() > p0.mean()


def test_same_profile_different_products_gives_different_recommendations(eng, raw):
    base = raw.iloc[3].to_dict()
    outs = {p: eng.score_one(dict(base, product_code=p)) for p in PRODUCT_CONFIG}
    assert len({(o["recommendedLimit"], o["recommendedRate"], o["maxTermMonths"]) for o in outs.values()}) == len(PRODUCT_CONFIG)


def test_limit_capped_by_product_and_non_negative(eng, raw):
    for r, (_, rw) in zip(eng.score_frame(raw), raw.iterrows()):
        assert 0 <= r["recommendedLimit"] <= PRODUCT_CONFIG[rw.product_code]["cap"]


def test_policy_flags_set_for_scenarios(eng, raw):
    base = raw.iloc[2].to_dict()
    cases = {"LIQUIDATION_IN_PROGRESS": dict(status="LIQUIDATING"), "ACCOUNT_BLOCKED": dict(fns_account_blocked=1),
             "MASS_ADDRESS": dict(fns_mass_address=1), "DISQUALIFIED_DIRECTOR": dict(fns_disqualified=1),
             "DELINQUENCY_90_PLUS": dict(bki_has_history=1, bki_overdue_90_cnt=1, bki_max_overdue_days=120)}
    for code, patch in cases.items():
        codes = {f["code"] for f in eng.score_one(dict(base, **patch))["policyFlags"]}
        assert code in codes and FLAG_DEFS[code][0] in ("CRITICAL", "WARNING")
    assert "NO_CREDIT_HISTORY" in {f["code"] for f in eng.score_one(dict(base, bki_has_history=0))["policyFlags"]}


def test_scoring_does_not_decide(eng, raw):
    """Граница ответственности: в ответе нет полей решения об одобрении."""
    r = eng.score_one(raw.iloc[0].to_dict())
    assert not {"decision", "approved", "status", "approve"} & set(r)


# ------------------------------------------------------------------ устойчивость
def test_extreme_and_missing_input_gives_warnings_not_exception(eng, raw):
    bad = raw.iloc[10].to_dict()
    bad.update(revenue_q8=1e15, requested_amount=np.nan, product_code="UNKNOWN_X", term_months=-3)
    r = eng.score_one(bad)
    assert 0 < r["pd"] < 1 and any(w.startswith("UNKNOWN_PRODUCT") for w in r["warnings"]) and any(w.startswith("INPUT_MISSING") for w in r["warnings"])


def test_empty_minimal_input_does_not_crash(eng):
    r = eng.score_one({"requested_amount": 1_000_000, "term_months": 12, "product_code": "EXPRESS"})
    assert r["recommendedLimit"] == 0 and len(r["warnings"]) > 0


def test_model_info(eng):
    info = eng.model_info(ROOT / "docs/ml/metrics.json")
    assert info["modelVersion"] == eng.version and len(info["ratingScale"]) >= 7 and "metrics" in info

Overwriting ../risk-scoring/tests/test_scoring.py


In [8]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(ROOT / "risk-scoring/tests/test_scoring.py")], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3500:], r.stderr[-1500:])

.......................                                                  [100%]
=============================== warnings summary ===============================
risk-scoring/tests/test_scoring.py::test_missing_columns_and_garbage_do_not_crash
  /home/arakteni/alfa-credit-scoring/risk-scoring/tests/test_scoring.py:56: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'abc' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
    bad.loc[bad.index[1], "requested_amount"] = "abc"

risk-scoring/tests/test_scoring.py::test_empty_minimal_input_does_not_crash
  /home/arakteni/alfa-credit-scoring/risk-scoring/riskscore/features.py:229: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_si